# Nhiệm vụ 6 — PCA + K-Means trên Development

Đối chiếu phần 6.1–6.12 của `M2/Ke_hoach_M2_Phan_cum_co_phieu.md`.
Giữ 8 feature, Global K=2, Robust Scaling riêng từng snapshot, không impute/clipping,
không đọc holdout, không đánh giá danh mục.

**Quy tắc PCA:** giữ số chiều cố định nhỏ nhất để **mọi snapshot development**
đạt cumulative explained variance ≥90%. Các model đã lưu dùng 4 chiều;
notebook kiểm chứng quy tắc này trên 15 tháng, không đổi số chiều từng tháng.

**Kết quả hiện hành v1:** kiểm chứng trực tiếp bộ model và artifacts đã sửa,
không fit lại và không cần bộ kết quả cũ.
Nguồn: `M2/artifacts/m2-task6-pca-kmeans-v1/`;
evaluation: `M2/artifacts/m2-evaluation-pca-kmeans/`;
model: `M2/models/pca_kmeans/`.
Lần chạy lại xác minh/tái lập cùng nội dung; không ghi đè evidence khác.


In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Tìm root bằng file config, không phụ thuộc tên thư mục hoặc máy cá nhân.
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'configs/experiments/m2_task6_pca.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Chạy notebook từ repository hoặc M2/notebooks.')
sys.path.insert(0, str(ROOT / 'src'))
from delta_t1.experiments.protocol import validate_protocol
CONFIG = json.loads((ROOT / 'configs/experiments/m2_task6_pca.json').read_text(encoding='utf-8'))
validate_protocol(CONFIG)
FEATURES = CONFIG['clustering']['features']
DEV_DATES = CONFIG['development_snapshots']
K = CONFIG['clustering']['k']
TASK_DIR = ROOT / 'M2/artifacts/m2-task6-pca-kmeans-v1'
EVAL_DIR = ROOT / 'M2/artifacts/m2-evaluation-pca-kmeans'
REPORT_DIR = ROOT / 'M2/reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Root:', ROOT)
print('Development:', DEV_DATES[0], '→', DEV_DATES[-1], '| Global K:', K)

def publish_csv(path, frame):
    content = frame.to_csv(index=False, lineterminator='\n').encode('utf-8')
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.read_bytes() != content:
        raise RuntimeError(f'CSV khác nội dung đã tồn tại: {path}; cần version mới.')
    if not path.exists():
        path.write_bytes(content)

def table_md(frame):
    def clean(value):
        return str(value).replace('|', '\\|').replace('\n', ' ')
    columns = frame.columns.tolist()
    lines = ['| ' + ' | '.join(map(clean, columns)) + ' |',
             '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in frame.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(map(clean, row)) + ' |')
    return '\n'.join(lines)

Root: C:\Users\HOME\Desktop\M2\Risk-Adjusted-Momentum-Clustering-m2-clustering-experiment
Development: 2023-11-30 → 2025-01-24 | Global K: 2


## 1. Nạp và kiểm chứng bộ kết quả hiện hành

Kiểm tra config, checksum, 15 model đã lưu và C8 development; không phụ thuộc artifacts cũ.

In [2]:
from datetime import datetime
from collections import Counter
from delta_t1.experiments.runner import prepare_m2_market_only_snapshots, m2_clustering_config
from delta_t1.evaluation.temporal_metrics import compare
from delta_t1.io import encoded, write_json, now

SOURCE_DIR = TASK_DIR
source_manifest = json.loads((SOURCE_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert source_manifest['status'] == 'complete'
assert source_manifest['artifact_id'] == 'm2-task6-pca-kmeans-v1'
assert source_manifest['n_snapshots'] == len(DEV_DATES)
assert source_manifest['config'] == CONFIG
source_run_id = source_manifest['lineage']['source_run_id']
for rel, expected in source_manifest['artifacts'].items():
    actual = hashlib.sha256((ROOT / rel).read_bytes()).hexdigest()
    assert actual == expected, rel

data_source, prepared = prepare_m2_market_only_snapshots(
    ROOT / 'artifacts/cafef_primary/cafef-c8-complete-only-v1', CONFIG)
validation = pd.DataFrame([{key: p[key] for key in
    ['snapshot_date', 'status', 'reason', 'eligible_count']} for p in prepared])
display(validation)
assert len(validation) == 15 and validation.status.eq('ready').all()
assert validation.eligible_count.ge(CONFIG['min_eligible_count']).all()
assert source_manifest['lineage']['c8_manifest_sha256'] == hashlib.sha256(
    (ROOT / 'artifacts/cafef_primary/cafef-c8-complete-only-v1/manifest.json').read_bytes()).hexdigest()
assert source_manifest['lineage']['c8_feature_sha256'] == data_source['outputs']['canonical/feature_snapshots.jsonl']
assert m2_clustering_config(CONFIG)['reduction']['n_components'] == CONFIG['clustering']['pca']['n_components']

original_assignments = pd.read_csv(SOURCE_DIR / 'assignments.csv')
assert not original_assignments.duplicated(['snapshot_date', 'security_id']).any()
original_profiles = pd.read_csv(SOURCE_DIR / 'cluster_profiles.csv')
diagnostics = pd.read_csv(SOURCE_DIR / 'diagnostics.csv').sort_values(['snapshot_date', 'k'])
models = {day: json.loads((SOURCE_DIR / f'models/{day}.json').read_text(encoding='utf-8')) for day in DEV_DATES}

,snapshot_date,status,reason,eligible_count
0,2023-11-30,ready,None,142
1,2023-12-29,ready,None,195
2,2024-01-31,ready,None,196
3,2024-02-29,ready,None,196
4,2024-03-29,ready,None,197
5,2024-04-26,ready,None,214
6,2024-05-31,ready,None,221
7,2024-06-28,ready,None,240
8,2024-07-31,ready,None,247
9,2024-08-30,ready,None,595


## 6.4–6.10 — Chiếu PCA, kiểm chứng assignments và hồ sơ gốc

PCA metadata được đọc tại `model['scaler']['_reduction']`.
Chiếu đúng theo mean và loadings đã fit; kiểm tra nhãn gần centroid nhất, inertia,
scaler median/IQR và `available_at` của từng row. Profile lấy trung bình **8 feature gốc
của thành viên thực tế**: không dùng inverse transform PCA bị cắt chiều để thay dữ liệu gốc.
Hungarian alignment được nối nhất quán từ tháng đầu; nhãn raw và aligned được tách riêng.

In [3]:
assignments, flat_profiles, score_rows, pca_rows, loading_rows = [], [], [], [], []
temporal_rows, transition_rows, drift_rows, scaler_rows = [], [], [], []
previous = None
aligned_ids = None
threshold = CONFIG['clustering']['pca']['explained_variance_threshold']
frozen_components = CONFIG['clustering']['pca']['n_components']

for prep in prepared:
    day, rows = prep['snapshot_date'], prep['rows']
    model = models[day]
    scaler, red = model['scaler'], model['scaler']['_reduction']
    assert red['n_components'] == frozen_components
    assert red['fit_scope'] == 'supplied_snapshot_only'
    assert model['config']['k'] == K and model['config']['features'] == FEATURES
    assert model['config']['scaling'] == 'robust_per_snapshot'
    assert model['config']['winsor_quantile'] == 0 and model['config']['clipping'] is False
    assert model['config']['reduction'] == m2_clustering_config(CONFIG)['reduction']
    assert all(model['config'][key] == CONFIG['clustering'][key] for key in ['seed', 'n_init', 'max_iter'])
    fit_time = datetime.fromisoformat(model['fit_as_of'])
    assert fit_time.date().isoformat() == day
    assert all(datetime.fromisoformat(row['available_at']) <= fit_time for row in rows)
    X = np.array([[row[f] for f in FEATURES] for row in rows], dtype=float)
    assert np.isfinite(X).all()
    center = np.array([scaler[f]['center'] for f in FEATURES])
    scale = np.array([scaler[f]['scale'] for f in FEATURES])
    q25, q75 = np.quantile(X, [.25, .75], axis=0)
    expected_scale = np.where(q75 == q25, 1., q75 - q25)
    assert np.allclose(center, np.median(X, axis=0)) and np.allclose(scale, expected_scale)
    assert np.all(scale > 0)
    for f, c, s in zip(FEATURES, center, scale):
        scaler_rows.append(dict(snapshot_date=day, feature=f, center=float(c), scale=float(s)))
    scaled = (X - center) / scale
    Z = (scaled - np.array(red['input_center'])) @ np.array(red['components']).T
    centers = np.array(model['centroids'])
    assert centers.shape == (K, frozen_components)
    labels = ((Z[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
    saved = original_assignments[original_assignments.snapshot_date.eq(day)].set_index('security_id')
    assert set(saved.index) == {row['security_id'] for row in rows}
    assert labels.tolist() == saved.loc[[r['security_id'] for r in rows], 'raw_cluster_id'].tolist()
    diag = diagnostics[diagnostics.snapshot_date.eq(day) & diagnostics.k.eq(K)].iloc[0]
    assert np.isclose(((Z - centers[labels]) ** 2).sum(), diag.inertia)

    profiles = []
    for cluster in range(K):
        members = X[labels == cluster]
        centroid = dict(zip(FEATURES, members.mean(axis=0).tolist()))
        assert np.allclose(centers[cluster], Z[labels == cluster].mean(axis=0))
        stored_profile = original_profiles[original_profiles.snapshot_date.eq(day)
            & original_profiles.raw_cluster_id.eq(cluster)].iloc[0]
        assert np.allclose(list(centroid.values()), [stored_profile[f] for f in FEATURES])
        profiles.append(dict(raw_cluster_id=cluster, centroid=centroid, size=len(members),
                             economic_rank=int(stored_profile.aligned_cluster_id)))
    snapshot = dict(rows=rows, labels=labels.tolist(), profiles=profiles, snapshot_date=day)
    old_ids = aligned_ids
    if previous is None:
        aligned_ids = {p['raw_cluster_id']: p['economic_rank'] for p in profiles}
    else:
        result = compare(previous, snapshot)
        assert result['mapping'] is not None
        aligned_ids = {current: old_ids[old] for current, old in result['mapping'].items()}
        temporal_rows.append(dict(from_date=previous['snapshot_date'], to_date=day,
            n_common=result['n_common'], ari=result['ari'], nmi=result['nmi'],
            persistence_probability=result['persistence_probability'], migration_rate=result['migration_rate'],
            entered_count=len(result['entered']), exited_count=len(result['exited']), status='OK'))
        for tr in result['transitions']:
            transition_rows.append(dict(from_date=previous['snapshot_date'], to_date=day,
                from_cluster=old_ids[tr['from_cluster']], to_cluster=old_ids[tr['to_cluster']],
                count=tr['count'], rate=tr['rate']))
        for current_id, previous_id in result['mapping'].items():
            for f in FEATURES:
                before = previous['profiles'][previous_id]['centroid'][f]
                after = profiles[current_id]['centroid'][f]
                drift_rows.append(dict(from_date=previous['snapshot_date'], to_date=day,
                    aligned_cluster_id=aligned_ids[current_id], feature=f,
                    value_from=before, value_to=after, delta=after-before))
    for row, raw_id, score in zip(rows, labels.tolist(), Z.tolist()):
        assert int(saved.loc[row['security_id'], 'aligned_cluster_id']) == aligned_ids[raw_id]
        assignments.append(dict(snapshot_date=day, security_id=row['security_id'], raw_cluster_id=raw_id,
            aligned_cluster_id=aligned_ids[raw_id], run_id=source_run_id,
            data_version=row['data_version'], pca_x=score[0], pca_y=score[1]))
        score_rows.append(dict(snapshot_date=day, security_id=row['security_id'],
            **{f'PC{i+1}': value for i, value in enumerate(score)}))
    for profile in profiles:
        flat_profiles.append(dict(snapshot_date=day, raw_cluster_id=profile['raw_cluster_id'],
            aligned_cluster_id=aligned_ids[profile['raw_cluster_id']], size=profile['size'],
            size_ratio=profile['size']/len(rows), **profile['centroid']))
    ratios = np.array(red['explained_variance_ratio'])
    cumulative = np.cumsum(ratios)
    assert cumulative[-1] >= threshold
    minimum = int(np.searchsorted(cumulative, threshold)) + 1
    pca_rows.append(dict(snapshot_date=day, n_eligible=len(rows), n_components=frozen_components,
        minimum_components_90=minimum, cumulative_explained_variance=float(cumulative[-1]),
        **{f'PC{i+1}_explained_variance': float(v) for i, v in enumerate(ratios)}))
    for i, component in enumerate(red['components'], 1):
        for f, value in zip(FEATURES, component):
            loading_rows.append(dict(snapshot_date=day, component=f'PC{i}', feature=f, loading=value))
    previous = snapshot

pca_summary = pd.DataFrame(pca_rows)
assert pca_summary.minimum_components_90.max() == frozen_components
assert len(assignments) == source_manifest['n_assignments'] == 5615
assert len(flat_profiles) == 30 and len(temporal_rows) == 14
assert len(transition_rows) == 56 and len(drift_rows) == 224
profile_frame = pd.DataFrame(flat_profiles)
assert profile_frame.columns.tolist() == ['snapshot_date', 'raw_cluster_id', 'aligned_cluster_id', 'size', 'size_ratio', *FEATURES]
for day in DEV_DATES:
    counts = Counter(a['aligned_cluster_id'] for a in assignments if a['snapshot_date'] == day)
    assert all(counts[int(p.aligned_cluster_id)] == int(p.size) for p in profile_frame[profile_frame.snapshot_date.eq(day)].itertuples())
display(pca_summary)
variance_components = []
for day in DEV_DATES:
    ratios = models[day]['scaler']['_reduction']['explained_variance_ratio']
    for component, (ratio, cumulative) in enumerate(zip(ratios, np.cumsum(ratios)), 1):
        variance_components.append(dict(snapshot_date=day, component=f'PC{component}',
            explained_variance=ratio, cumulative_explained_variance=float(cumulative)))
display(pd.DataFrame(variance_components))
display(pd.DataFrame(loading_rows).pivot(index=['snapshot_date', 'component'], columns='feature', values='loading'))
display(profile_frame)
display(diagnostics[diagnostics.k.eq(K)])

,snapshot_date,n_eligible,n_components,minimum_components_90,cumulative_explained_variance,PC1_explained_variance,PC2_explained_variance,PC3_explained_variance,PC4_explained_variance
0,2023-11-30,142,4,2,0.973850,0.794927,0.105567,0.059869,0.013486
1,2023-12-29,195,4,3,0.968639,0.799032,0.091027,0.045416,0.033164
2,2024-01-31,196,4,3,0.951956,0.708338,0.143302,0.060154,0.040162
3,2024-02-29,196,4,3,0.952492,0.757335,0.114309,0.049231,0.031616
4,2024-03-29,197,4,4,0.936424,0.620535,0.202923,0.073638,0.039328
5,2024-04-26,214,4,3,0.962517,0.684699,0.169513,0.073832,0.034474
6,2024-05-31,221,4,3,0.950950,0.674163,0.167950,0.072149,0.036689
7,2024-06-28,240,4,3,0.955743,0.701086,0.145547,0.069061,0.040049
8,2024-07-31,247,4,3,0.943748,0.608031,0.207912,0.085278,0.042527
9,2024-08-30,595,4,2,0.997031,0.870791,0.120676,0.003412,0.002153


,snapshot_date,component,explained_variance,cumulative_explained_variance
0,2023-11-30,PC1,0.794927,0.794927
1,2023-11-30,PC2,0.105567,0.900494
2,2023-11-30,PC3,0.059869,0.960363
3,2023-11-30,PC4,0.013486,0.973850
4,2023-12-29,PC1,0.799032,0.799032
5,2023-12-29,PC2,0.091027,0.890059
6,2023-12-29,PC3,0.045416,0.935475
7,2023-12-29,PC4,0.033164,0.968639
8,2024-01-31,PC1,0.708338,0.708338
9,2024-01-31,PC2,0.143302,0.851640


feature                  beta_126  liquidity_21   mdd_126   mom_126    mom_21  \
snapshot_date component                                                         
2023-11-30    PC1        0.053346      0.996709 -0.012446  0.020113  0.040924   
              PC2        0.096785     -0.054070  0.058055  0.565167  0.310229   
              PC3       -0.431994      0.051829  0.527950  0.200264 -0.404504   
              PC4        0.037365     -0.012109  0.077902 -0.079886  0.721925   
2023-12-29    PC1        0.057277      0.992510 -0.023380  0.049380  0.039874   
              PC2        0.087437     -0.106043  0.058175  0.519528  0.346258   
              PC3       -0.434972      0.049400  0.659536  0.197140 -0.095119   
              PC4       -0.088645     -0.004280 -0.053687 -0.378851  0.818191   
2024-01-31    PC1        0.070533      0.989606 -0.019278  0.051887  0.047119   
              PC2        0.041513     -0.115010  0.153422  0.390958  0.638968   
              PC3        0.354757     -0.070307 -0.465232  0.027497 -0.384009   
              PC4       -0.277611      0.021374  0.476240  0.504970 -0.473778   
2024-02-29    PC1        0.067583      0.991050 -0.024845  0.052125  0.025751   
              PC2        0.063486     -0.105335  0.121483  0.486207  0.442891   
              PC3        0.347149     -0.044497 -0.606037 -0.117147 -0.034057   
              PC4       -0.409401      0.061289  0.239698  0.065548  0.051716   
2024-03-29    PC1        0.109728      0.973523 -0.032874  0.104772  0.070488   
              PC2        0.102193     -0.203456  0.078191  0.516345  0.412168   
              PC3       -0.279327      0.048808  0.715740  0.203413 -0.109766   
              PC4        0.228165     -0.031237 -0.178693 -0.401237  0.638351   
2024-04-26    PC1        0.073837      0.992746  0.003652  0.061380 -0.011129   
              PC2       -0.060285     -0.059695  0.320975  0.514949  0.399360   
              PC3        0.337449     -0.076006 -0.458417  0.214369 -0.313905   
              PC4       -0.242411      0.053945 -0.420033 -0.216778  0.620962   
2024-05-31    PC1        0.100945      0.977527  0.017668  0.105414  0.050012   
              PC2        0.085979     -0.177171  0.079850  0.505138  0.520224   
              PC3       -0.331648      0.004644  0.684674  0.201548 -0.235429   
              PC4       -0.063686      0.063437  0.009501 -0.248051  0.725899   
2024-06-28    PC1        0.067793      0.991577  0.024071  0.061585  0.008395   
              PC2       -0.073093     -0.087005  0.207718  0.520367  0.442494   
              PC3        0.262025     -0.017758 -0.547555  0.093050 -0.196817   
              PC4       -0.254436      0.071195 -0.351074 -0.306779  0.695894   
2024-07-31    PC1        0.085099      0.990169  0.020558  0.061895  0.014934   
              PC2       -0.053912     -0.088888  0.298443  0.442626  0.549332   
              PC3        0.232634     -0.044707 -0.442872  0.321098 -0.415993   
              PC4        0.113440      0.040583 -0.361949 -0.304106  0.559735   
2024-08-30    PC1        0.010795      0.999874  0.003084 -0.001736  0.008414   
              PC2       -0.006704      0.001375  0.002603  0.664296  0.133797   
              PC3        0.087938     -0.010061  0.098322 -0.177837  0.971583   
              PC4       -0.016037      0.006193 -0.645619  0.001257  0.114899   
2024-09-30    PC1        0.007375      0.999943  0.002722  0.000282  0.003128   
              PC2       -0.068035     -0.004492  0.140509  0.504219  0.587830   
              PC3        0.000920     -0.002175  0.084079 -0.440718  0.673527   
              PC4       -0.037166      0.005040 -0.653802 -0.035609  0.278001   
2024-10-31    PC1        0.005575      0.999954  0.003330  0.000637 -0.001198   
              PC2       -0.039451     -0.003043  0.093548  0.397769  0.653073   
              PC3       -0.056041      0.004471 -0.217903 -0.491964  0.633738   
              PC4       -0.1194

,snapshot_date,raw_cluster_id,aligned_cluster_id,size,size_ratio,mom_21,mom_63,mom_126,mom_252,vol_63,mdd_126,beta_126,liquidity_21
0,2023-11-30,0,0,9,0.063380,0.092225,-0.013171,0.084650,0.423786,0.410883,-0.238019,1.267496,2.012277e+11
1,2023-11-30,1,1,133,0.936620,0.050468,-0.061309,0.015675,0.203257,0.334853,-0.215860,0.616263,8.895114e+09
2,2023-12-29,0,1,179,0.917949,0.031677,0.010857,0.021818,0.264321,0.327619,-0.235505,0.762964,1.450657e+10
3,2023-12-29,1,0,16,0.082051,0.062248,0.134621,0.196020,0.501271,0.394096,-0.268225,1.471359,3.324347e+11
4,2024-01-31,0,1,179,0.913265,0.009575,0.070983,-0.019414,0.188632,0.269506,-0.230780,0.740147,1.426615e+10
5,2024-01-31,1,0,17,0.086735,0.030720,0.206539,0.098224,0.412317,0.304670,-0.276454,1.487746,2.951742e+11
6,2024-02-29,0,1,179,0.913265,0.046654,0.084319,0.034134,0.284181,0.265063,-0.219312,0.718767,1.668228e+10
7,2024-02-29,1,0,17,0.086735,0.074429,0.164255,0.144411,0.657540,0.252574,-0.266550,1.496977,3.591518e+11
8,2024-03-29,0,1,182,0.923858,0.032841,0.094751,0.083572,0.310896,0.269400,-0.174081,0.684781,2.609622e+10
9,2024-03-29,1,0,15,0.076142,0.053765,0.188808,0.208734,0.642257,0.271505,-0.193871,1.426396,4.920194e+11


,calinski_harabasz,cluster_balance,cluster_sizes,converged,davies_bouldin,inertia,k,silhouette,snapshot_date
0,225.635234,0.067669,"{""0"": 9, ""1"": 133}",True,0.536114,1219.015572,2,0.775353,2023-11-30
7,417.106858,0.089385,"{""0"": 179, ""1"": 16}",True,0.472406,1267.010612,2,0.771666,2023-12-29
14,261.530664,0.094972,"{""0"": 179, ""1"": 17}",True,0.558519,1360.147611,2,0.719430,2024-01-31
21,361.527940,0.094972,"{""0"": 179, ""1"": 17}",True,0.482618,1295.238398,2,0.745119,2024-02-29
28,190.663829,0.082418,"{""0"": 182, ""1"": 15}",True,0.590633,1257.003284,2,0.676387,2024-03-29
35,261.893710,0.070000,"{""0"": 200, ""1"": 14}",True,0.547559,1535.166794,2,0.728588,2024-04-26
42,232.343692,0.072816,"{""0"": 15, ""1"": 206}",True,0.573273,1543.374428,2,0.704280,2024-05-31
49,274.715119,0.048035,"{""0"": 229, ""1"": 11}",True,0.515502,1963.065124,2,0.763995,2024-06-28
56,176.314178,0.073913,"{""0"": 230, ""1"": 17}",True,0.716695,2009.521186,2,0.659505,2024-07-31
63,926.687158,0.036585,"{""0"": 574, ""1"": 21}",True,0.472977,121106.779768,2,0.906888,2024-08-30


## 6.11–6.12 — Kiểm chứng bộ v1 hiện hành có lineage và checksum

Temporal inputs được tạo từ cùng assignments/profiles đã kiểm chứng ở trên.
`value_from` và `value_to` là tâm gốc thật; mọi transition dùng nhãn aligned xuyên chuỗi.
Trạng thái complete chỉ được ghi sau khi tất cả kiểm tra đã qua.

In [4]:
def immutable_bytes(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != content:
            raise RuntimeError(f'Artifact khác nội dung đã tồn tại: {path}. Tạo version mới.')
    else:
        path.write_bytes(content)

def csv_bytes(frame):
    return frame.to_csv(index=False, lineterminator='\n').encode('utf-8')

def jsonl_bytes(rows):
    return b''.join(encoded(row) + b'\n' for row in rows)

outputs = {}
for name, records in [('assignments', assignments), ('pca_scores', score_rows), ('pca_diagnostics', pca_rows)]:
    outputs[TASK_DIR / f'{name}.csv'] = csv_bytes(pd.DataFrame(records))
    outputs[TASK_DIR / f'{name}.jsonl'] = jsonl_bytes(records)
outputs[TASK_DIR / 'cluster_profiles.csv'] = csv_bytes(profile_frame)
outputs[TASK_DIR / 'profiles.jsonl'] = jsonl_bytes(flat_profiles)
outputs[TASK_DIR / 'diagnostics.csv'] = (SOURCE_DIR / 'diagnostics.csv').read_bytes()
outputs[TASK_DIR / 'pca_loadings.csv'] = csv_bytes(pd.DataFrame(loading_rows))
outputs[TASK_DIR / 'scaler_parameters.csv'] = csv_bytes(pd.DataFrame(scaler_rows))
outputs[TASK_DIR / 'skipped_snapshots.jsonl'] = b''
outputs[TASK_DIR / 'config.json'] = encoded(CONFIG) + b'\n'
outputs[EVAL_DIR / 'cluster_profiles.csv'] = csv_bytes(profile_frame)
for name, records in [('temporal_stability', temporal_rows), ('transition_matrices', transition_rows), ('centroid_drift', drift_rows)]:
    outputs[EVAL_DIR / f'{name}.csv'] = csv_bytes(pd.DataFrame(records))
for day in DEV_DATES:
    original = (SOURCE_DIR / f'models/{day}.json').read_bytes()
    outputs[TASK_DIR / f'models/{day}.json'] = original
    outputs[ROOT / f'M2/models/pca_kmeans/{day}.json'] = original
for path, content in outputs.items():
    immutable_bytes(path, content)
# Preserve historical lineage and the owner-requested storage migration metadata.
new_manifest = dict(source_manifest)
new_manifest.update(config=CONFIG, n_snapshots=len(prepared), n_assignments=len(assignments),
    artifacts={path.relative_to(ROOT).as_posix(): hashlib.sha256(content).hexdigest()
               for path, content in outputs.items()})
immutable_bytes(TASK_DIR / 'manifest.json', encoded(new_manifest) + b'\n')
print('PASS: 15 model, 5.615 assignments, 30 profiles, scores PC1–PC4, 14 cặp aligned.')

evr = pca_summary.cumulative_explained_variance
conclusion = f"""# Báo cáo Nhiệm vụ 6 — PCA + K-Means

Nguồn model: `{source_run_id}`. Bộ kiểm chứng/xuất lại: `m2-task6-pca-kmeans-v1`.
Bộ kết quả đã sửa được lưu tại đường dẫn v1 theo yêu cầu người dùng.
Fitted model được tái dùng và kiểm chứng, không huấn luyện lại.

## A — Thực thi và kiểm chứng
15/15 snapshot; universe {validation.eligible_count.min()}–{validation.eligible_count.max()} mã;
5.615 nhãn khớp dự đoán từ model, 30 profiles trên 8 feature gốc.
Global K={K}; Robust Scaling độc lập; seed=42, n_init=10, max_iter=300.

## B — Quy tắc PCA đã kiểm chứng
Số chiều cố định {frozen_components} là max của số chiều tối thiểu đạt 90% trên từng development snapshot.
Phương sai giữ lại: median {evr.median():.4%}, min {evr.min():.4%}, max {evr.max():.4%}.
Không tùy ý đổi số chiều từng tháng và không sử dụng holdout.

## C — Bàn giao nhiệm vụ 7–9
Đủ assignments, PC1–PC4 scores, loadings, PCA diagnostics, scaler parameters,
13-cột cluster_profiles, quality diagnostics và ba temporal inputs aligned.
Model ở `M2/models/pca_kmeans/`; manifest kiểm tra checksum của toàn bộ bộ xuất.
Kết quả chỉ mô tả development market-only, không phải dynamic clustering hoặc bằng chứng sinh lợi.
"""
display(Markdown(conclusion))
for name in ['Bao_cao_M2_Nhiem_vu_6_PCA_KMeans.md']:
    (REPORT_DIR / name).write_text(conclusion.rstrip() + '\n', encoding='utf-8')

PASS: 15 model, 5.615 assignments, 30 profiles, scores PC1–PC4, 14 cặp aligned.


# Báo cáo Nhiệm vụ 6 — PCA + K-Means

Nguồn model: `experiment-20261003T113748Z-acb3a2e7`. Bộ kiểm chứng/xuất lại: `m2-task6-pca-kmeans-v1`.
Bộ kết quả đã sửa được lưu tại đường dẫn v1 theo yêu cầu người dùng.
Fitted model được tái dùng và kiểm chứng, không huấn luyện lại.

## A — Thực thi và kiểm chứng
15/15 snapshot; universe 142–780 mã;
5.615 nhãn khớp dự đoán từ model, 30 profiles trên 8 feature gốc.
Global K=2; Robust Scaling độc lập; seed=42, n_init=10, max_iter=300.

## B — Quy tắc PCA đã kiểm chứng
Số chiều cố định 4 là max của số chiều tối thiểu đạt 90% trên từng development snapshot.
Phương sai giữ lại: median 96.8639%, min 93.6424%, max 99.8398%.
Không tùy ý đổi số chiều từng tháng và không sử dụng holdout.

## C — Bàn giao nhiệm vụ 7–9
Đủ assignments, PC1–PC4 scores, loadings, PCA diagnostics, scaler parameters,
13-cột cluster_profiles, quality diagnostics và ba temporal inputs aligned.
Model ở `M2/models/pca_kmeans/`; manifest kiểm tra checksum của toàn bộ bộ xuất.
Kết quả chỉ mô tả development market-only, không phải dynamic clustering hoặc bằng chứng sinh lợi.


## Kết luận A/B/C từ số liệu đã chạy

## A — Thực thi và kiểm chứng
15/15 snapshot; universe 142–780 mã;
5.615 nhãn khớp dự đoán từ model, 30 profiles trên 8 feature gốc.
Global K=2; Robust Scaling độc lập; seed=42, n_init=10, max_iter=300.

## B — Quy tắc PCA đã kiểm chứng
Số chiều cố định 4 là max của số chiều tối thiểu đạt 90% trên từng development snapshot.
Phương sai giữ lại: median 96.8639%, min 93.6424%, max 99.8398%.
Không tùy ý đổi số chiều từng tháng và không sử dụng holdout.

## C — Bàn giao nhiệm vụ 7–9
Đủ assignments, PC1–PC4 scores, loadings, PCA diagnostics, scaler parameters,
13-cột cluster_profiles, quality diagnostics và ba temporal inputs aligned.
Model ở `M2/models/pca_kmeans/`; manifest kiểm tra checksum của toàn bộ bộ xuất.
Kết quả chỉ mô tả development market-only, không phải dynamic clustering hoặc bằng chứng sinh lợi.